In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The <a href="https://en.wikipedia.org/wiki/DPLL_algorithm">Davis-Putnam Algorithm</a> with the Jeroslow-Wang Heuristic

This notebook implements the algorithm of Davis and Putnam.  Further details about this algorithm are provided in the lecture notes.

We use the types `Literal`, `Clause`, and `CNF` that are defined in the notebook `04-CNF.ipynb`:
```
type Literal =
    | Pos of string   // p
    | Neg of string   // ¬p

type Clause = Set<Literal>
type CNF    = Set<Clause>
```

In [2]:
importNotebook "04-CNF.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb

The function `complement` takes one argument:
- `l` is a literal.

It returns the complement of `l`.
If $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

As we are working with clauses that result from transforming given formulas into *conjunctive normal form* and these clauses do not contain $\top$ or $\bot$, we don't have to bother with $\top$ or $\bot$ in this function.

In [3]:
let complement (l: Literal) : Literal =
    match l with
    | Pos p -> Neg p
    | Neg p -> Pos p

The function `extractVariable` takes one argument:
- `l` is a literal.

It returns the variable of the literal `l`.
If $p$ is a propositional variable, we have the following: 
* $\texttt{extractVariable}(p) = p$,
* $\texttt{extractVariable}(\neg p) = p$.

In [4]:
let extractVariable (l: Literal) : string =
    match l with
    | Pos p -> p
    | Neg p -> p

The function `arb` takes one argument:
- `S` is a non-empty set.

It returns an arbitrary element from the set `S`.  As F# sets are ordered, we simply return the smallest element.

In [5]:
let arb (S: Set<'T>) : 'T =
    Set.minElement S

The function `selectLiteral` takes three arguments:
- `Clauses` is a set of clauses,
- `Variables` is the set of all variables occurring in `Clauses`,
- `UsedVars` is the set of variables that have already been used in case distinctions.

It returns a literal $l$ such that the variable of $l$ does not occur in the set `UsedVars`.  It uses the *Jeroslow-Wang heuristic* to choose the literal. The Jeroslow-Wang heuristic $\texttt{JW}(l)$ of a literal $l$ in a set of clauses is defined as follows:
$$ \texttt{JW}(\textrm{Clauses}, l) = \sum\limits_{\{\,C \in \texttt{Clauses}\;\mid\; l \in C\;\}} \frac{1}{\;2^{|C|}\;} $$ 
Here, $|C|$ denotes the number of literals in the clause $C$.  The idea is to choose a literal that occurs in many short clauses and therefore subsumes a lot of clauses.  The function returns the literal with the highest score.

**Implementation:** The map `Scores` assigns a score to every candidate literal.  For every variable $p$ that has not been used, both literals $p$ and $\neg p$ are candidates and their scores are initialized with $0$.  Then we iterate over all clauses $C$ and add $2^{-|C|}$ to the score of every candidate literal occurring in $C$.  The expression `2.0 ** -(float C.Count)` computes $2^{-|C|}$ as a floating point number.  Finally, `Seq.maxBy snd` returns the pair `(l, score)` with the highest score and `fst` extracts the literal `l` from this pair.

In [6]:
let selectLiteral (Clauses: CNF) (Variables: Set<string>) (UsedVars: Set<string>) : Literal =
    let mutable Scores : Map<Literal, float> = Map.empty
    for p in Variables do
        if not (UsedVars.Contains p) then
            Scores <- Scores.Add(Pos p, 0.0).Add(Neg p, 0.0)
    for C in Clauses do
        for l in C do
            if Scores.ContainsKey l then
                Scores <- Scores.Add(l, Scores[l] + 2.0 ** -(float C.Count))
    fst (Seq.maxBy snd (Map.toSeq Scores))

The function `Seq.maxBy f xs` returns the element `x` of the sequence `xs` such that `f x` is maximal.  For example, if we have a map that assigns numbers to strings, we can find the key with the maximal value as follows.  `Map.toSeq` converts a map into a sequence of key-value pairs.

In [7]:
let D = Map [ ("a", 3); ("b", 5); ("c", 1) ]
fst (Seq.maxBy snd (Map.toSeq D))

"b"

The function `reduce` takes two arguments:
- `Clauses` is a set of clauses,
- `l` is a literal such that the unit clause $\{l\}$ is an element of `Clauses`.

It performs all unit cuts and all unit subsumptions on clauses of the set `Clauses` that are possible using the unit clause $\{\mathtt{l}\}$.  The resulting set of clauses is returned.  Mathematically, the function `reduce` is defined as follows:
$$\texttt{reduce}(\texttt{Clauses},l)  := 
 \Bigl\{\, C \backslash \bigl\{\overline{\,l\,}\bigr\} \;|\; C \in \texttt{Clauses} \wedge \overline{\,l\,} \in C \,\Bigr\} 
       \,\cup\, \Bigl\{\, C \in \texttt{Clauses} \mid \overline{\,l\,} \not\in C \wedge l \not\in C \Bigr\} \cup \bigl\{\{l\}\bigr\}.
$$

**Implementation:** A direct implementation of this formula would rebuild the complete set of clauses.  For F# sets this is expensive, since inserting a clause into a set of clauses requires comparing the clause with other clauses.  However, only those clauses that contain $l$ or $\overline{\,l\,}$ are affected by `reduce`.  Therefore, we first compute the set `Affected` of these clauses.  The clauses in `Affected` are removed from `Clauses` and replaced by
- the clauses $C \backslash \bigl\{\overline{\,l\,}\bigr\}$ for those clauses $C \in \texttt{Affected}$ that do not contain $l$, and
- the unit clause $\{l\}$.

Clauses that contain $l$ are subsumed by the unit clause $\{l\}$ and are therefore dropped.  The result is the same set of clauses as specified by the formula given above.

In [8]:
let reduce (Clauses: CNF) (l: Literal) : CNF =
    let lBar     = complement l
    let Affected = Set.filter (fun (C: Clause) -> C.Contains l || C.Contains lBar) Clauses
    let Reduced  = set [ for C in Affected do if not (C.Contains l) then C.Remove lBar ]
    (Clauses - Affected) + Reduced + set [ set [ l ] ]

The function `saturate` takes one argument:
- `Clauses` is a set of clauses.

It returns the set of those clauses that can be derived from `Clauses` via repeated applications of unit cuts or unit subsumptions.

**Implementation:** The mutable variable `S` contains the current set of clauses, `Units` contains the unit clauses of `S` that have not yet been used, and `Used` contains the unit clauses that have already been used for reductions.  As long as there is a unit clause that has not been used, we use it to reduce `S`.

In [9]:
let saturate (Clauses: CNF) : CNF =
    let mutable S     = Clauses
    let mutable Units = Set.filter (fun (C: Clause) -> C.Count = 1) S
    let mutable Used  : Set<Clause> = Set.empty
    while not Units.IsEmpty do  // iterate as long as we derive new unit clauses
        let unit = arb Units
        Used  <- Used.Add unit
        let l = arb unit
        S     <- reduce S l
        Units <- Set.filter (fun (C: Clause) -> C.Count = 1) S - Used
    S

The function `solve` takes one argument:
- `Clauses` is a set of clauses.

The function tries to compute a variable assignment that satisfies all clauses in `Clauses`.  If this is successful, a set of unit clauses is returned.  This set of unit clauses does not contain any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If the set `Clauses` is unsatisfiable, then the set $\bigl\{\{\}\bigr\}$ is returned instead.

The real work is done by the function `solveRecursive`.  This function takes three arguments:
* `Clauses` is the set of clauses that is to be solved,
* `Variables` is the set of all variables occurring in `Clauses`,
* `UsedVars`  is the set of those variables that have already been used in case distinctions.

**Implementation:** First, `Clauses` is saturated using unit cuts and unit subsumptions.  If the resulting set contains the empty clause, it is unsatisfiable.  If it contains only unit clauses, these unit clauses form a solution.  Otherwise, we use the Jeroslow-Wang heuristic to select the most promising literal $l$ whose variable has not been used yet and perform a case distinction: First we try to solve the problem with the additional unit clause $\{l\}$.  If this fails, we try the unit clause $\{\overline{\,l\,}\}$.

In [10]:
let rec solveRecursive (Clauses: CNF) (Variables: Set<string>) (UsedVars: Set<string>) : CNF =
    let S      = saturate Clauses
    let Falsum = set [ Set.empty ]
    if S.Contains Set.empty then                         // S is inconsistent
        Falsum
    elif Set.forall (fun (C: Clause) -> C.Count = 1) S then  // S is trivial
        S
    else
        // use the Jeroslow-Wang heuristic to select the most promising literal l
        let l       = selectLiteral S Variables UsedVars
        let lBar    = complement l
        let newUsed = UsedVars.Add (extractVariable l)
        let Result  = solveRecursive (S.Add (set [ l ])) Variables newUsed
        if Result <> Falsum then
            Result
        else
            solveRecursive (S.Add (set [ lBar ])) Variables newUsed

In [11]:
let solve (Clauses: CNF) : CNF =
    let Variables = set [ for C in Clauses do for l in C -> extractVariable l ]
    solveRecursive Clauses Variables Set.empty

The function `solutionToString` takes two arguments:
- `S` is a set of clauses,
- `Solution` is the result of `solve S`.

It returns a string describing the solution:  If `S` is unsolvable, the string says so.  Otherwise, the string lists the values of all variables.  The function `cnfToString` is defined in the notebook `04-CNF.ipynb`.

The auxiliary function `unitToString` takes a unit clause and returns a string of the form `p ↦ true` or `p ↦ false`.

In [12]:
let unitToString (C: Clause) : string =
    match arb C with
    | Pos p -> p + " ↦ true"
    | Neg p -> p + " ↦ false"

let solutionToString (S: CNF) (Solution: CNF) : string =
    if Solution = set [ Set.empty ] then
        cnfToString S + " is unsolvable"
    else
        "{ " + String.concat ", " [ for C in Solution -> unitToString C ] + " }"

In [13]:
let c1 : Clause = set [ Pos "r"; Pos "p"; Pos "s" ]
let c2 : Clause = set [ Pos "r"; Pos "s" ]
let c3 : Clause = set [ Pos "p"; Pos "q"; Pos "s" ]
let c4 : Clause = set [ Neg "p"; Neg "q" ]
let c5 : Clause = set [ Neg "p"; Pos "s"; Neg "r" ]
let c6 : Clause = set [ Pos "p"; Neg "q"; Pos "r" ]
let c7 : Clause = set [ Neg "r"; Neg "s"; Pos "q" ]
let c8 : Clause = set [ Neg "p"; Neg "s" ]
let c9 : Clause = set [ Pos "p"; Neg "r"; Neg "q" ]
let c0 : Clause = set [ Neg "p"; Pos "r"; Pos "q"; Neg "s" ]
let S1 = set [ c0; c1; c2; c3; c4; c5; c6; c7; c8; c9 ]
printfn "%s" (solutionToString S1 (solve S1))

{ s ↦ true, p ↦ false, q ↦ false, r ↦ false }

In [14]:
let c10 : Clause = set [ Pos "p"; Pos "r"; Pos "q"; Neg "s" ]
let S2 = set [ c0; c1; c2; c3; c4; c5; c6; c7; c8; c9; c10 ]
printfn "%s" (solutionToString S2 (solve S2))

{{p, q, r, ¬s}, {p, q, s}, {p, r, s}, {p, r, ¬q}, {p, ¬q, ¬r}, {q, r, ¬p, ¬s}, {q, ¬r, ¬s}, {r, s}, {s, ¬p, ¬r}, {¬p, ¬q}, {¬p, ¬s}} is unsolvable

In [15]:
let S3 = set [ set [ Pos "r"; Pos "p"; Pos "s" ]
               set [ Pos "r"; Pos "s" ]
               set [ Pos "q"; Pos "p"; Pos "s" ]
               set [ Neg "p"; Neg "q" ]
               set [ Neg "p"; Pos "s"; Neg "r" ]
               set [ Pos "p"; Neg "q"; Pos "r" ]
               set [ Neg "r"; Neg "s"; Pos "q" ]
               set [ Pos "p"; Pos "q"; Pos "r"; Pos "s" ]
               set [ Pos "r"; Neg "s"; Pos "q" ]
               set [ Pos "s"; Neg "r"; Neg "q" ]
               set [ Pos "s"; Neg "r" ]
               set [ Pos "r"; Neg "s" ] ]
solve S3

set [set [Pos "q"]; set [Pos "r"]; set [Pos "s"]; set [Neg "p"]]